# NIDS Project - Results Analysis

**Author:** Hamza Arshad

This notebook analyzes experimental results and generates figures for the IEEE paper.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.visualization.plots import (
    plot_quality_latency_tradeoff,
    plot_confusion_matrix,
    plot_per_class_recall,
    plot_model_comparison_table
)

%matplotlib inline

## 1. Load All Results

In [ ]:
DATASET = 'cicids2017'  # or 'unsw-nb15'

# Load results from all models
models_dir = project_root / f'models/{DATASET}'
results = {}

for model_dir in models_dir.iterdir():
    if model_dir.is_dir():
        test_metrics_path = model_dir / 'test_metrics.json'
        latency_path = model_dir / 'latency.json'
        
        if test_metrics_path.exists() and latency_path.exists():
            with open(test_metrics_path) as f:
                test_metrics = json.load(f)
            with open(latency_path) as f:
                latency = json.load(f)
            
            results[model_dir.name] = {
                'metrics': test_metrics,
                'latency': latency
            }

print(f"Loaded results for {len(results)} models:")
for model_name in results.keys():
    print(f"  - {model_name}")

## 2. Quality Metrics Comparison

In [ ]:
# Create comparison table
comparison_data = []

for model_name, data in results.items():
    comparison_data.append({
        'Model': model_name.replace('_', ' ').title(),
        'Accuracy': data['metrics']['accuracy'],
        'Macro-F1': data['metrics']['macro_f1'],
        'Macro Precision': data['metrics']['macro_precision'],
        'Macro Recall': data['metrics']['macro_recall'],
        'Weighted F1': data['metrics']['weighted_f1']
    })

df_quality = pd.DataFrame(comparison_data)
df_quality = df_quality.sort_values('Macro-F1', ascending=False)

print("Quality Metrics Comparison:")
print("=" * 100)
print(df_quality.to_string(index=False))

# Highlight best model for RQ1
best_model = df_quality.iloc[0]
print(f"\n✓ Best Model (Macro-F1): {best_model['Model']} ({best_model['Macro-F1']:.4f})")

## 3. Latency Metrics Comparison

In [ ]:
# Create latency comparison table
latency_data = []

for model_name, data in results.items():
    latency_data.append({
        'Model': model_name.replace('_', ' ').title(),
        'Mean Latency (ms)': data['latency']['per_flow_inference']['mean_time_ms'],
        'P95 Latency (ms)': data['latency']['per_flow_inference']['p95_time_ms'],
        'P99 Latency (ms)': data['latency']['per_flow_inference']['p99_time_ms'],
        'Throughput (fps)': data['latency']['throughput']['throughput_fps'],
        'Model Size (MB)': data['latency']['model_size']['size_mb']
    })

df_latency = pd.DataFrame(latency_data)
df_latency = df_latency.sort_values('Mean Latency (ms)')

print("Latency Metrics Comparison:")
print("=" * 100)
print(df_latency.to_string(index=False))

# Highlight fastest model
fastest_model = df_latency.iloc[0]
print(f"\n✓ Fastest Model: {fastest_model['Model']} ({fastest_model['Mean Latency (ms)']:.4f} ms)")

## 4. Quality-Latency Tradeoff (RQ2)

In [ ]:
# Merge quality and latency for tradeoff plot
df_tradeoff = df_quality[['Model', 'Macro-F1']].merge(
    df_latency[['Model', 'Mean Latency (ms)']],
    on='Model'
)
df_tradeoff.rename(columns={'Mean Latency (ms)': 'Latency (ms)'}, inplace=True)

# Plot quality-latency tradeoff
plot_quality_latency_tradeoff(
    df_tradeoff,
    title=f'Quality-Latency Tradeoff on {DATASET.upper()}'
)

print("\nThis plot answers RQ2: How do models compare on quality AND speed?")

## 5. Per-Class Recall Comparison (RQ1 - Rare Attacks)

In [ ]:
# Extract per-class recall for all models
per_class_recall = {}

for model_name, data in results.items():
    model_display_name = model_name.replace('_', ' ').title()
    per_class_recall[model_display_name] = {
        class_name: metrics['recall']
        for class_name, metrics in data['metrics']['per_class'].items()
    }

# Plot per-class recall comparison
plot_per_class_recall(
    per_class_recall,
    title=f'Per-Class Recall Comparison on {DATASET.upper()}'
)

print("\nThis plot answers RQ1: Can CNN match tree ensembles on rare attack classes?")

## 6. Focus on Rare Classes

In [ ]:
# Identify rare classes and compare performance
from src.data.data_loader import load_splits

splits = load_splits(DATASET)
y_train = splits['y_train']

unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)

# Get class names
preprocessor_path = project_root / f'data/processed/{DATASET}_preprocessor/metadata.json'
with open(preprocessor_path) as f:
    metadata = json.load(f)
    class_names = list(metadata['label_mapping'].keys())

# Identify rare classes (< 5%)
rare_threshold = 0.05
rare_classes = []

for cls_idx, count in zip(unique, counts):
    proportion = count / total
    if proportion < rare_threshold:
        rare_classes.append(class_names[cls_idx])

print(f"Rare Classes (< {rare_threshold*100}% of dataset):")
print("=" * 70)

if rare_classes:
    # Compare performance on rare classes
    rare_performance = []
    
    for model_name, recall_dict in per_class_recall.items():
        rare_recalls = [recall_dict[cls] for cls in rare_classes if cls in recall_dict]
        if rare_recalls:
            avg_rare_recall = np.mean(rare_recalls)
            rare_performance.append({
                'Model': model_name,
                'Avg Rare Class Recall': avg_rare_recall
            })
    
    df_rare = pd.DataFrame(rare_performance)
    df_rare = df_rare.sort_values('Avg Rare Class Recall', ascending=False)
    
    print(df_rare.to_string(index=False))
    
    best_on_rare = df_rare.iloc[0]
    print(f"\n✓ Best on Rare Classes: {best_on_rare['Model']} ({best_on_rare['Avg Rare Class Recall']:.4f})")
    print("\nThis directly answers RQ1!")
else:
    print("No rare classes found in this dataset.")

## 7. Confusion Matrices

In [ ]:
# Plot confusion matrix for each model
for model_name, data in results.items():
    cm = np.array(data['metrics']['confusion_matrix'])
    class_names_local = list(data['metrics']['per_class'].keys())
    
    plot_confusion_matrix(
        cm,
        class_names_local,
        model_name.replace('_', ' ').title(),
        normalize=True
    )

## 8. Answering Research Questions

### RQ1: Can a compact 1D-CNN match or beat RF and XGBoost on rare attack classes?

In [ ]:
# Compare models on macro-F1 and rare class recall
rq1_summary = df_quality[['Model', 'Macro-F1']].merge(
    df_rare[['Model', 'Avg Rare Class Recall']] if rare_classes else pd.DataFrame(),
    on='Model',
    how='left'
)

print("RQ1 Analysis: Model Performance on Rare Attacks")
print("=" * 70)
print(rq1_summary.to_string(index=False))

# Determine answer
cnn_models = [m for m in rq1_summary['Model'] if 'cnn' in m.lower() or '1d' in m.lower()]
if cnn_models:
    cnn_row = rq1_summary[rq1_summary['Model'].isin(cnn_models)].iloc[0]
    baseline_rows = rq1_summary[~rq1_summary['Model'].isin(cnn_models)]
    
    best_baseline_f1 = baseline_rows['Macro-F1'].max()
    
    print(f"\nAnswer to RQ1:")
    if cnn_row['Macro-F1'] >= best_baseline_f1:
        print(f"✓ YES! {cnn_row['Model']} matches/beats baselines on macro-F1")
    else:
        diff = best_baseline_f1 - cnn_row['Macro-F1']
        print(f"✗ NO. {cnn_row['Model']} is {diff:.4f} points below best baseline")
        print(f"   However, check latency tradeoff below...")

### RQ2: How do models compare on quality AND speed?

In [ ]:
# Full comparison
rq2_summary = df_quality[['Model', 'Macro-F1']].merge(
    df_latency[['Model', 'Mean Latency (ms)', 'Model Size (MB)']],
    on='Model'
)

print("RQ2 Analysis: Quality-Latency-Size Tradeoff")
print("=" * 70)
print(rq2_summary.to_string(index=False))

print("\nAnswer to RQ2:")
print("See the quality-latency tradeoff plot above.")
print("\nKey observations:")
print("1. Which model offers the best quality?")
print("2. Which model is fastest?")
print("3. Which model offers the best quality-latency tradeoff?")
print("4. Is there a model that dominates on both axes?")

## 9. Save Results for Paper

In [ ]:
# Save all comparison tables
output_dir = project_root / f'experiments/results/{DATASET}'
output_dir.mkdir(parents=True, exist_ok=True)

df_quality.to_csv(output_dir / 'quality_comparison.csv', index=False)
df_latency.to_csv(output_dir / 'latency_comparison.csv', index=False)
df_tradeoff.to_csv(output_dir / 'quality_latency_tradeoff.csv', index=False)

if rare_classes:
    df_rare.to_csv(output_dir / 'rare_class_performance.csv', index=False)

print(f"✓ All results saved to {output_dir}")
print("\nUse these tables in your IEEE paper!")